# Thyroid Cancer Recurrence Classification with XGBoost

This notebook presents a thyroid cancer recurrence classification workflow using XGBoost, Information Gain feature selection, and grid search. Experiment controls and visible analysis remain in the notebook; reusable implementation is provided by `train_xgboost_thyroid_classification.py`.

Data source: the included `Thyroid_Diff.csv` is the UCI Differentiated Thyroid Cancer Recurrence dataset. Its citation, dataset page, and CC BY 4.0 license are documented in `README.md`.

Workflow summary:

| Item | Setting |
|---|---|
| Target / y | `Recurred` |
| Input / X | Tabular clinical features |
| Task | Binary classification |
| Validation | Default: plain K-Fold Cross Validation with 10 folds |
| Feature selection | Entropy-based Information Gain fitted within each training fold |
| Main metric | Accuracy by default, higher is better |
| Extra metrics | Precision, recall, specificity, F1-score, MCC, ROC-AUC, PR-AUC, and Brier score |
| Model | `XGBClassifier` |
| Data checks | Raw audit -> preprocessing -> cleaned data validation -> EDA |
| Compute device | Notebook default: `cpu`; can be changed to `auto` or `gpu` |

Step 1 is the experiment control panel. A top-to-bottom run reproduces preprocessing, fold-local feature selection, default and tuned experiments, out-of-fold evaluation, the selected confusion matrix, and saved artifacts. Fold-wise Information Gain rankings are reused across hyperparameter combinations.


## 0. Notebook Setup

Imports the display libraries and reusable functions from `train_xgboost_thyroid_classification.py`.


In [ ]:
%matplotlib inline

from argparse import Namespace
from pathlib import Path
import json
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

import train_xgboost_thyroid_classification as thyroid_backend
importlib.reload(thyroid_backend)

from train_xgboost_thyroid_classification import (
    METRIC_DISPLAY_DECIMALS,
    format_decimal,
    validate_args,
    set_reproducible_seed,
    load_dataset,
    raw_data_audit,
    feature_role_table,
    encode_feature_dataframe,
    build_encoded_modeling_dataframe,
    preprocessing_summary_table,
    clean_classification_dataset,
    split_features_target,
    encode_target,
    build_param_grid,
    xgboost_fixed_params_for_display,
    format_xgboost_device_report,
    xgboost_device_info,
    make_kfold,
    build_fold_numbers,
    resolve_search_workers,
    resolve_parallel_fit_threads,
    run_grid_search,
    run_default_models_by_feature_count,
    finalize_classification_results,
    build_prediction_table,
    save_confusion_matrix_plot,
    cleanup_old_output_runs,
    round_metric_columns,
)


### Notebook Display Helpers

These notebook display helpers keep tables, short summaries, and plots consistent. They only change presentation; model training and metric calculations stay unchanged. Tables remain available without row/column truncation for Data Wrangler.


In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


def display_dataframe(frame, decimals=METRIC_DISPLAY_DECIMALS):
    displayed = frame.copy()
    numeric_cols = displayed.select_dtypes(include=["number"]).columns
    displayed[numeric_cols] = displayed[numeric_cols].round(decimals)
    return displayed


from IPython.display import display as _ipython_display
pd.set_option("display.float_format", lambda value: format_decimal(value))


def display(*objects, **kwargs):
    prepared = [
        display_dataframe(item) if isinstance(item, pd.DataFrame) else item
        for item in objects
    ]
    return _ipython_display(*prepared, **kwargs)


def metric_display_dataframe(frame):
    return display_dataframe(round_metric_columns(frame, decimals=METRIC_DISPLAY_DECIMALS))


def display_summary(title, items, note=None):
    lines = [f"**{title}**", ""]
    for label, value in items:
        if isinstance(value, (float, np.floating)):
            value = format_decimal(value)
        elif isinstance(value, (list, tuple)):
            value = ", ".join(str(item) for item in value)
        lines.append(f"- **{label}:** {value}")
    if note:
        lines.extend(["", note])
    display(Markdown("\n".join(lines)))


def apply_notebook_grid(ax, settings):
    if settings.grid:
        ax.grid(True)


## 1. Workflow Configuration

Edit this cell for ordinary experiments; the backend does not need to be changed.

| Group | Settings | Purpose |
|---|---|---|
| Data setup | `data`, `target_col`, `positive_class`, `exclude_features` | Select the dataset, target setup, and optional feature exclusions. |
| Validation and feature selection | `cv_splits`, `selection_metric`, `ig_top_k` | Control K-Fold evaluation, model selection, and Information Gain feature-count experiments. |
| Grid search | `colsample_bytree`, `learning_rates`, `max_depths`, `n_estimators`, `subsamples` | Test the selected XGBoost classification hyperparameter combinations. |
| Manual-check settings | `tree_method`, `base_score`, `gamma`, `min_child_weight`, `reg_alpha`, `reg_lambda`, `scale_pos_weight`, `max_bin` | Keep split construction, initial prediction, class weighting, and regularization choices explicit. |
| Runtime and resume | `device`, `n_jobs`, `search_workers`, `seed`, `run_name`, `progress_every` | Control compute resources, repeatability, and checkpoint frequency. |

The XGBoost estimator hyperparameter names are singular (`learning_rate`, `max_depth`, `n_estimators`, `subsample`, and `colsample_bytree`); the notebook uses plural list names for grid values. `ig_top_k` controls the fold-local Information Gain feature-count experiment. `search_workers=-1` automatically uses logical CPU threads in CPU mode and one worker in CUDA mode; CUDA also requires `tree_method="hist"`. Plot settings stay in their respective plotting cells.


In [ ]:
args = Namespace(
    # Dataset and target.
    data=Path("Thyroid_Diff.csv"),
    target_col="Recurred",
    exclude_features=[],
    positive_class="Yes",

    # Validation, Information Gain, and model selection.
    cv_splits=10,
    selection_metric="accuracy",
    ig_top_k=["all", 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20],

    # XGBoost hyperparameter grid.
    learning_rates=[0.01, 0.05, 0.1, 0.2, 0.3],
    max_depths=[4, 5, 6, 7, 8],
    n_estimators=[100, 200, 300, 400, 500],
    subsamples=[0.6, 0.7, 0.8, 0.9, 1.0],
    colsample_bytree=[0.6, 0.7, 0.8, 0.9, 1.0],

    # Reproducibility and compute.
    seed=42,
    device="cpu",
    tree_method="exact",
    n_jobs=-1,
    search_workers=-1,

    # Fixed workflow choices outside the grid. None keeps the XGBoost default.
    base_score=0.5,
    gamma=None,
    min_child_weight=None,
    reg_alpha=None,
    reg_lambda=None,
    scale_pos_weight=None,
    max_bin=None,

    # Checkpoint and output settings.
    run_name="notebook_xgboost_classification_checkpoint",
    # Show one progress message per 500 completed combinations.
    progress_every=500,
    verbose=1,
    output_dir=Path("outputs"),
    keep_runs=1,
    show_plot=False,
    prepare_only=False,
)

validate_args(args)
set_reproducible_seed(args.seed)

output_dir = args.output_dir / args.run_name
hyperparameter_combinations = int(np.prod([
    len(args.learning_rates),
    len(args.max_depths),
    len(args.n_estimators),
    len(args.subsamples),
    len(args.colsample_bytree),
]))
screening_fits = len(args.ig_top_k) * args.cv_splits
grid_combinations = len(args.ig_top_k) * hyperparameter_combinations
grid_fits = grid_combinations * args.cv_splits
final_oof_fits = args.cv_splits
total_fits = screening_fits + grid_fits + final_oof_fits
effective_search_workers = resolve_search_workers(args.search_workers, args.device)
parallel_fit_threads = resolve_parallel_fit_threads(args.n_jobs, effective_search_workers)

display_summary(
    "Run plan",
    [
        ("Dataset", args.data.name),
        ("Target / positive class", f"{args.target_col} / {args.positive_class}"),
        ("Validation", f"K-Fold with {args.cv_splits} folds"),
        ("IG feature counts", args.ig_top_k),
        ("Feature-count screening fits", screening_fits),
        ("Grid combinations", grid_combinations),
        ("Grid-search fits", grid_fits),
        ("Final out-of-fold prediction fits", final_oof_fits),
        ("Total planned model fits", total_fits),
        ("Compute", f"{args.device.upper()} / {args.tree_method}"),
        ("Parallel workers", effective_search_workers),
        ("Threads per parallel fit", parallel_fit_threads),
        ("Selection metric", args.selection_metric),
        ("Checkpoint file", output_dir / "grid_search_progress.csv"),
    ],
    "Completed combinations are saved to the checkpoint so an interrupted search can continue.",
)


## 2. Raw Data Audit

This stage inspects the selected CSV before modeling. The audit makes row count, column types, missing values, duplicate rows, and class balance visible before preprocessing.


### 2.1 Raw Data Overview

The overview summarizes the main quality signals from the raw file before any row or column is changed.


In [ ]:
df = load_dataset(args.data, args.target_col)

raw_audit_df = raw_data_audit(df, args.target_col)
missing_df = df.isna().sum().reset_index()
missing_df.columns = ["column", "missing_values"]
dtype_df = df.dtypes.astype(str).reset_index()
dtype_df.columns = ["column", "dtype"]
unique_df = df.nunique(dropna=False).reset_index()
unique_df.columns = ["column", "unique_values"]
summary_df = dtype_df.merge(missing_df, on="column").merge(unique_df, on="column")

display_summary(
    "Raw dataset overview",
    [(str(row["item"]).replace("_", " " ).title(), row["value"]) for _, row in raw_audit_df.iterrows()],
    "This check shows dataset size, duplicate and missing rows, and the original target balance.",
)

display(Markdown("**Column summary**"))
display(summary_df)


### 2.2 Raw Dataset Table

The full raw dataset is displayed after the audit so the original input structure can still be inspected directly.


In [ ]:
display(Markdown("**Raw dataset**"))
display(df)


### 2.3 Raw Data Issues to Fix

The raw audit identifies the issues handled before modeling. Exact duplicate rows are removed by keeping the first occurrence, rows with missing values are removed, and feature availability concerns can be tested through `exclude_features`.


## 3. Data Preprocessing

This stage applies the cleaning rules and prepares the classification inputs. The target is encoded, each feature is assigned a modeling role, and the encoded feature table is created for Information Gain and XGBoost.


### 3.1 Apply Cleaning Rules

Exact duplicates and missing rows are handled first. The target is then encoded with the configured positive class.


In [ ]:
clean_df, preprocessing_stats = clean_classification_dataset(df)
preprocessing_summary_df = preprocessing_summary_table(preprocessing_stats)
df = clean_df

x, y_raw = split_features_target(df, args.target_col, args.exclude_features)
y_encoded, target_metadata = encode_target(y_raw, args.positive_class)
feature_roles_df = feature_role_table(x)
encoded_x, encoded_feature_roles_df = encode_feature_dataframe(x)
encoded_modeling_df = build_encoded_modeling_dataframe(encoded_x, y_encoded, args.target_col)

label_mapping_df = pd.DataFrame(
    [{"class_label": key, "encoded_value": value} for key, value in target_metadata["label_mapping"].items()]
)

display_summary(
    "Preprocessing action summary",
    [(str(row["item"]).replace("_", " " ).title(), row["value"]) for _, row in preprocessing_summary_df.iterrows()],
    "These values show how many rows were removed before encoding.",
)
display(Markdown("**Target label mapping**"))
display(label_mapping_df)


### 3.2 Feature Encoding Roles

Input features are encoded according to their modeling role: numeric features stay numeric, binary and ordinal features use explicit numeric mappings, and nominal features are expanded with one-hot columns.


In [ ]:
display(Markdown("**Original feature encoding roles**"))
display(feature_roles_df)

display(Markdown("**Encoded feature roles**"))
display(encoded_feature_roles_df)


Feature-role rationale:

- `T`, `N`, `M`, and `Stage` are ordered cancer staging variables.
- `Risk` and `Response` are ordered clinical summary variables based on the 2015 American Thyroid Association differentiated thyroid cancer guideline. `Response` can be tested as nominal in a sensitivity run when the analysis avoids imposing clinical order.
- Binary fields are mapped to `0/1` because each has two observed states.
- Multi-category fields without a safe monotonic order are one-hot encoded.

References used for the role decision are listed in the README.


### 3.3 Encoding Output Check

This check records the table shapes after cleaning and encoding, including the increase in feature count caused by one-hot encoded columns.


In [ ]:
display_summary(
    "Encoding output shape",
    [
        ("Cleaned features before encoding", f"{x.shape[0]} rows x {x.shape[1]} columns"),
        ("Encoded features", f"{encoded_x.shape[0]} rows x {encoded_x.shape[1]} columns"),
        ("Encoded data with target", f"{encoded_modeling_df.shape[0]} rows x {encoded_modeling_df.shape[1]} columns"),
    ],
    "The row count stays aligned; one-hot encoding can increase the number of feature columns.",
)


## 4. Cleaned Data Validation

This stage checks the cleaned and encoded tables before cross-validation. This check confirms that the modeling data is complete, aligned, and ready for feature selection.


### 4.1 Data Quality Check

The cleaned dataset is checked for remaining missing values, duplicate rows, and feature-target alignment.


In [ ]:
quality_check_df = pd.DataFrame([
    {"check": "cleaned_rows", "value": len(df)},
    {"check": "cleaned_columns", "value": df.shape[1]},
    {"check": "remaining_missing_values", "value": int(df.isna().sum().sum())},
    {"check": "remaining_duplicate_rows", "value": int(df.duplicated().sum())},
    {"check": "encoded_feature_rows", "value": int(encoded_x.shape[0])},
    {"check": "encoded_target_rows", "value": int(y_encoded.shape[0])},
])

cleaned_column_summary_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_values": df.isna().sum().values,
    "unique_values": df.nunique(dropna=False).values,
})

display(Markdown("**Cleaned data quality check**"))
display(quality_check_df)

display(Markdown("**Cleaned column summary**"))
display(cleaned_column_summary_df)


### 4.2 Validated Modeling Dataset

The cleaned table and encoded modeling table are displayed with the target column placed last for easier inspection.


In [ ]:
cleaned_display_df = df[[col for col in df.columns if col != args.target_col] + [args.target_col]].copy()

display(Markdown("**Cleaned dataset**"))
display(cleaned_display_df)
display(Markdown("**Encoded modeling dataset**"))
display(encoded_modeling_df)


## 5. Exploratory Data Analysis

EDA focuses on class balance, numeric ranges, categorical cardinality, and encoded feature scale. These checks give context for the validation results and for later sensitivity experiments.


### 5.1 Target Distribution

The target distribution is checked before K-Fold validation because class imbalance affects how the reported classification and probability metrics are interpreted.


In [ ]:
target_counts = df[args.target_col].value_counts().rename_axis("class_label").reset_index(name="count")
target_counts["percentage"] = target_counts["count"] / target_counts["count"].sum() * 100

plot_cfg = Namespace(figsize=(8, 5), grid=True, dpi=160)

display(Markdown("**Target class distribution**"))
display(metric_display_dataframe(target_counts))

fig, ax = plt.subplots(figsize=plot_cfg.figsize, dpi=plot_cfg.dpi)
ax.bar(target_counts["class_label"], target_counts["count"])
ax.set_title("Target Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
apply_notebook_grid(ax, plot_cfg)
fig.tight_layout()
plt.show()


### 5.2 Numeric Feature Summary

Numeric summaries make the scale and spread of numeric input columns visible before model training.


In [ ]:
numeric_cols = [col for col in df.columns if col != args.target_col and pd.api.types.is_numeric_dtype(df[col])]

if numeric_cols:
    display(Markdown("**Numeric feature summary**"))
    display(metric_display_dataframe(df[numeric_cols].describe().T))
else:
    display(Markdown("No numeric input columns were detected before encoding."))


### 5.3 Categorical Feature Summary

Categorical summaries show cardinality and dominant values for the original categorical inputs before encoding.


In [ ]:
categorical_cols = [col for col in df.columns if col not in numeric_cols + [args.target_col]]

categorical_summary_rows = []
for col in categorical_cols:
    counts = df[col].value_counts(dropna=False)
    categorical_summary_rows.append({
        "feature": col,
        "unique_values": df[col].nunique(dropna=False),
        "most_common_value": counts.index[0],
        "most_common_count": int(counts.iloc[0]),
    })

categorical_summary_df = pd.DataFrame(categorical_summary_rows)
display(Markdown("**Categorical feature summary**"))
display(categorical_summary_df)


### 5.4 Encoded Feature Summary

The encoded feature summary checks the final numeric columns used by Information Gain and XGBoost.


In [ ]:
encoded_summary_df = encoded_x.describe().T.reset_index().rename(columns={"index": "encoded_feature"})

display(Markdown("**Encoded feature summary**"))
display(metric_display_dataframe(encoded_summary_df))


## 6. K-Fold Validation and Information Gain

This stage sets up K-Fold Cross Validation. Information Gain is computed only from the training subset in each fold, so feature selection remains part of the cross-validation workflow.

### K-Fold Summary

Plain K-Fold does not force class proportions to match across folds, so the notebook displays the class distribution in each fold's test portion.


In [ ]:
cv = make_kfold(args.cv_splits, args.seed)
fold_numbers = build_fold_numbers(encoded_x, y_encoded, cv)

fold_summary_rows = []
for fold in range(1, args.cv_splits + 1):
    fold_mask = fold_numbers == fold
    labels = target_metadata["encoder"].inverse_transform(y_encoded[fold_mask])
    counts = pd.Series(labels).value_counts().rename_axis("class_label").reset_index(name="count")
    counts.insert(0, "fold", fold)
    counts["percentage"] = counts["count"] / counts["count"].sum() * 100
    fold_summary_rows.append(counts)

fold_distribution_df = pd.concat(fold_summary_rows, ignore_index=True)
fold_size_df = pd.DataFrame({
    "fold": range(1, args.cv_splits + 1),
    "test_samples_in_fold": [int((fold_numbers == fold).sum()) for fold in range(1, args.cv_splits + 1)],
})

display(Markdown("**Fold size summary**"))
display(fold_size_df)

display(Markdown("**Class distribution in each fold test portion**"))
display(metric_display_dataframe(fold_distribution_df))


## 7. XGBoost Feature Screening and Grid Search

This stage first screens the configured Information Gain feature counts with default XGBoost. The full hyperparameter grid is then evaluated for all encoded features and every configured numeric feature count.


### 7.1 Compute Runtime Report

The report records the device configuration used by XGBoost training. Feature encoding and Information Gain preparation run on CPU, while XGBoost fit and predict use CuPy GPU arrays when `device="gpu"` is selected.


In [ ]:
device_info = xgboost_device_info(args.device, args.tree_method)
display(Markdown(
    "**Compute runtime report**\n\n```text\n"
    + format_xgboost_device_report(device_info)
    + "\n```"
))


### 7.2 Information Gain Feature-Count Screening

Default XGBoost evaluates all configured feature-count choices. Information Gain is fitted only on each training fold, and the numeric top-k with the highest mean accuracy is retained. Ties use F1-score, PR-AUC, ROC-AUC, then the smaller feature count.


In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
feature_roles_df.to_csv(output_dir / "feature_encoding_roles.csv", index=False)
encoded_modeling_df.to_csv(output_dir / "encoded_classification_dataset.csv", index=False)
encoded_feature_roles_df.to_csv(output_dir / "encoded_feature_encoding_roles.csv", index=False)

default_results_df, default = run_default_models_by_feature_count(
    args=args,
    x=encoded_x,
    y=y_encoded,
    target_metadata=target_metadata,
)
default_screened_top_k = int(default["ig_top_k"])
grid_top_k_values = list(args.ig_top_k)

display_summary(
    "Feature-count preparation complete",
    [
        ("Configured feature setups", len(default_results_df)),
        ("Best default IG feature count", default_screened_top_k),
        ("Grid feature setups", grid_top_k_values),
        ("Selection metric", args.selection_metric),
    ],
    "The cached fold rankings are reused while the full grid evaluates every configured feature count.",
)

The checkpointed grid tests every hyperparameter combination for every configured feature setup. The Information Gain ranking is computed once from each fold's training subset and reused throughout the search. Four setups are retained in the visible summary: default and tuned XGBoost with all features, the best default IG feature count, and the independently selected best tuned IG feature count.

In [ ]:
results_df, best = run_grid_search(
    args=args,
    x=encoded_x,
    y=y_encoded,
    target_metadata=target_metadata,
    output_dir=output_dir,
    grid_top_k_values=grid_top_k_values,
    fold_ig_cache=default.get("fold_ig_cache"),
)

finalized = finalize_classification_results(
    args=args,
    x=encoded_x,
    y=y_encoded,
    target_metadata=target_metadata,
    best=best,
    default_results_df=default_results_df,
)

best = finalized["best"]
default_vs_grid_by_feature_count_df = finalized["default_vs_grid_by_feature_count"]
classification_main_summary_df = finalized["main_summary"]
classification_supporting_summary_df = finalized["supporting_summary"]

default_vs_grid_by_feature_count_df.to_csv(output_dir / "default_vs_grid_search_by_feature_count.csv", index=False)
classification_main_summary_df.to_csv(output_dir / "classification_main_summary.csv", index=False)
classification_supporting_summary_df.to_csv(output_dir / "classification_supporting_summary.csv", index=False)
pd.DataFrame([best["best_params"]]).to_csv(output_dir / "best_grid_search_parameters.csv", index=False)
best["selected_features_by_fold"].to_csv(output_dir / "best_selected_features_by_fold.csv", index=False)
round_metric_columns(best["selected_feature_frequency"]).to_csv(output_dir / "best_selected_feature_frequency.csv", index=False)
round_metric_columns(best["fold_metrics"]).to_csv(output_dir / "best_fold_metrics.csv", index=False)

display(Markdown("**Accuracy-focused model summary**"))
display(classification_main_summary_df)
display(Markdown("**Supporting classification metrics**"))
display(classification_supporting_summary_df)


## 8. Cross-Validated Predictions

The table contains out-of-fold predictions. Each row is predicted by a model trained without that row, based on its assigned fold.


In [ ]:
predictions_df = build_prediction_table(
    x=encoded_x,
    y=y_encoded,
    y_pred=best["y_cv_pred"],
    y_proba=best["y_cv_proba"],
    target_metadata=target_metadata,
    fold_numbers=fold_numbers,
)
predictions_df.to_csv(output_dir / "best_cv_predictions.csv", index=False)
display(Markdown("**Out-of-fold predictions**  \nEach row was predicted by a model that did not train on that row."))
display(predictions_df)


## 9. Confusion Matrix

The confusion matrix summarizes out-of-fold classification errors from the final selected setup: the selected grid-search hyperparameters and Information Gain feature count.


In [ ]:
save_confusion_matrix_plot(
    y_true=y_encoded,
    y_pred=best["y_cv_pred"],
    class_labels=target_metadata["classes"],
    output_path=output_dir / "best_confusion_matrix.png",
    show_plot=False,
)

display(Image(filename=str(output_dir / "best_confusion_matrix.png")))


## 10. Limitations

The detailed limitations and follow-up directions are documented in `README.md`. The constraints most relevant when interpreting this run are:

- feature counts and hyperparameters are selected and reported with the same non-nested K-Fold procedure;
- the 383-row dataset and plain K-Fold split make results sensitive to fold assignment and class balance;
- Information Gain is associative rather than causal, and some clinical summary features may be unavailable at the intended prediction time.


## 11. Final Results and Output Files

The four principal experiment setups, reported metrics, selected configuration, and run location are consolidated below. Feature-count and fold-level details remain available in the modeling section and generated files.


In [ ]:
metadata = {
    "data_path": str(args.data),
    "target_col": args.target_col,
    "exclude_features": args.exclude_features,
    "cv_splits": args.cv_splits,
    "positive_class": args.positive_class,
    "selection_metric": args.selection_metric,
    "preprocessing_stats": preprocessing_stats,
    "feature_encoding_roles": feature_roles_df.to_dict("records"),
    "encoded_feature_encoding_roles": encoded_feature_roles_df.to_dict("records"),
    "encoded_feature_count": int(encoded_x.shape[1]),
    "seed": args.seed,
    "device": args.device,
    "tree_method": args.tree_method,
    "n_jobs": args.n_jobs,
    "search_workers": args.search_workers,
    "effective_search_workers": effective_search_workers,
    "run_name": args.run_name,
    "progress_every": args.progress_every,
    "compute_device": device_info,
    "feature_count_screening_options": list(args.ig_top_k),
    "default_screened_ig_top_k": default_screened_top_k,
    "selected_grid_ig_top_k": best["best_params"].get("ig_top_k"),
    "grid_feature_setups": grid_top_k_values,
    "param_grid": build_param_grid(args, grid_top_k_values),
    "feature_count_screening_fits": screening_fits,
    "grid_search_combinations": grid_combinations,
    "grid_search_fits": grid_fits,
    "final_oof_prediction_fits": final_oof_fits,
    "total_planned_model_fits": total_fits,
    "fixed_xgboost_params": xgboost_fixed_params_for_display(args),
    "target_metadata": {key: value for key, value in target_metadata.items() if key != "encoder"},
    "best_params": best["best_params"],
    "selected_features": best["selected_features"],
    "checkpoint_signature": best.get("checkpoint_metadata", {}).get("signature"),
}
with (output_dir / "run_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, default=str)
removed_runs = cleanup_old_output_runs(args.output_dir, output_dir, args.keep_runs)

final_classification_main_df = classification_main_summary_df.copy()
final_classification_main_df.insert(
    0,
    "Rank",
    final_classification_main_df["Accuracy"].rank(method="min", ascending=False).astype(int),
)
final_classification_main_df = final_classification_main_df.sort_values(["Rank", "Result"]).reset_index(drop=True)
for metric in ["Accuracy", "Precision", "Recall", "Specificity", "F1-score"]:
    final_classification_main_df[metric] = pd.to_numeric(final_classification_main_df[metric], errors="coerce") * 100
final_classification_main_df = final_classification_main_df.rename(columns={
    metric: f"{metric} (%)"
    for metric in ["Accuracy", "Precision", "Recall", "Specificity", "F1-score"]
})

final_classification_supporting_df = classification_supporting_summary_df.copy()
final_classification_supporting_df = final_classification_supporting_df.merge(
    final_classification_main_df[["Rank", "Result"]],
    on="Result",
    how="left",
).sort_values(["Rank", "Result"]).reset_index(drop=True)
for metric in ["ROC-AUC", "PR-AUC"]:
    final_classification_supporting_df[metric] = pd.to_numeric(final_classification_supporting_df[metric], errors="coerce") * 100
final_classification_supporting_df = final_classification_supporting_df.rename(columns={
    "ROC-AUC": "ROC-AUC (%)",
    "PR-AUC": "PR-AUC (%)",
})
final_classification_supporting_df = final_classification_supporting_df[[
    "Rank", "Result", "MCC", "ROC-AUC (%)", "PR-AUC (%)", "Brier score", "Setup"
]]

winner = final_classification_main_df.iloc[0]
selected_parameter_text = ", ".join(f"{key}={value}" for key, value in best["best_params"].items())
display(Markdown("**Primary classification metrics**"))
display(metric_display_dataframe(final_classification_main_df))
display(Markdown("**Probability and supporting metrics**"))
display(metric_display_dataframe(final_classification_supporting_df))
display(Markdown("**Information Gain feature selection frequency**"))
display(metric_display_dataframe(best["selected_feature_frequency"]))
display(Markdown(
    "`selected_in_folds` counts how many training folds selected each "
    "feature. For example, 10 of 10 folds means the feature was selected "
    "in every fold."
))

display_summary(
    "Key findings",
    [
        ("Highest-accuracy setup", f"{winner['Result']} ({format_decimal(winner['Accuracy (%)'])}%)"),
        ("Confusion-matrix setup", "Best grid search + best IG feature count"),
        ("Selected hyperparameters", selected_parameter_text),
        ("Selected feature count", len(best["selected_features"])),
        ("Validation", f"{args.cv_splits}-fold K-Fold"),
        ("Output folder", str(output_dir)),
    ],
    "Feature-count results, fold metrics, predictions, the confusion matrix, and metadata are stored in the output folder.",
)
if removed_runs:
    display(Markdown(f"Removed {len(removed_runs)} older output run(s) according to `keep_runs`."))
